# Phase 2: Analysis


## Data Set Up
Combining data, basic data cleaning, and loading data.

In [4]:
# Needed setup/imports
import pandas as pd
import pandas as pd
import json
import os
import re
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
# Load and combine all user data
base_path = "data"

all_history = []
all_playlists = []
all_library_tracks = []

for user_folder in os.listdir(base_path):
    user_path = os.path.join(base_path, user_folder)
    
    if not os.path.isdir(user_path):
        continue

    # Streaming data
    stream_files = glob(os.path.join(user_path, "StreamingHistory_music_*.json"))
    user_history = []

    for file in stream_files:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)
            user_history.extend(data)

    if user_history:
        df_hist = pd.DataFrame(user_history)
        df_hist["user_id"] = user_folder
        all_history.append(df_hist)

    # Playlist data
    playlist_files = glob(os.path.join(user_path, "Playlist*.json"))
    playlist_tracks = []

    for file in playlist_files:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        playlists = data.get("playlists", data)

        for pl in playlists:
            items = pl.get("items", [])

            for item in items:
                track = item.get("track")

                # Handle missing tracks
                if not track or not isinstance(track, dict):
                    continue

                playlist_tracks.append({
                    "artistName": track.get("artistName"),
                    "trackName": track.get("trackName"),
                    "playlistName": pl.get("name"),
                    "user_id": user_folder
                })

    if playlist_tracks:
        df_pl = pd.DataFrame(playlist_tracks)
        all_playlists.append(df_pl)

    # Library data
    lib_path = os.path.join(user_path, "YourLibrary.json")

    if os.path.exists(lib_path):
        with open(lib_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        tracks = data.get("tracks", [])

        for t in tracks:
            all_library_tracks.append({
                "artistName": t.get("artist"),
                "trackName": t.get("track"),
                "user_id": user_folder
            })

# Combine all users data

history_all = pd.concat(all_history, ignore_index=True)
playlist_all = pd.concat(all_playlists, ignore_index=True)

# Handling for empty library tracks
if all_library_tracks:
    library_tracks = pd.DataFrame(all_library_tracks)
else:
    library_tracks = pd.DataFrame(columns=["artistName", "trackName", "user_id"])

print("Streaming shape:", history_all.shape)
print("Playlist shape:", playlist_all.shape)
print("Library shape:", library_tracks.shape)

Streaming shape: (205849, 5)
Playlist shape: (18368, 4)
Library shape: (6851, 3)


## Data preprocessing
Clean and standardize text, preprocessing/feature engineering for analysis

### Cleaning and Matching

In [6]:
# Cleaning and standardizing text
def clean_text(text):
    if pd.isna(text):
        return None
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9 ]", "", text)
    return text

# Apply cleaning
history_all["artist_clean"] = history_all["artistName"].apply(clean_text)
history_all["track_clean"] = history_all["trackName"].apply(clean_text)

playlist_all["artist_clean"] = playlist_all["artistName"].apply(clean_text)
playlist_all["track_clean"] = playlist_all["trackName"].apply(clean_text)

library_tracks["artist_clean"] = library_tracks["artistName"].apply(clean_text)
library_tracks["track_clean"] = library_tracks["trackName"].apply(clean_text)


# Create match keys
history_all["match_key"] = history_all["artist_clean"] + "_" + history_all["track_clean"]
playlist_all["match_key"] = playlist_all["artist_clean"] + "_" + playlist_all["track_clean"]
library_tracks["match_key"] = library_tracks["artist_clean"] + "_" + library_tracks["track_clean"]

playlist_keys = playlist_all.groupby("user_id")["match_key"].apply(set).to_dict()
library_keys = library_tracks.groupby("user_id")["match_key"].apply(set).to_dict()

# Time preprocessing
history_all["endTime"] = pd.to_datetime(history_all["endTime"])
history_all["hour"] = history_all["endTime"].dt.hour
history_all["day_of_week"] = history_all["endTime"].dt.day_name()
history_all["month"] = history_all["endTime"].dt.month

# Determine curation level
def get_curation_level(row):
    key = row["match_key"]
    user = row["user_id"]
    
    in_playlist = key in playlist_keys.get(user, set())
    in_library = key in library_keys.get(user, set())
    
    if in_playlist and in_library:
        return 3
    elif in_playlist:
        return 2
    elif in_library:
        return 1
    else:
        return 0

history_all["curation_level"] = history_all.apply(get_curation_level, axis=1)

# Map curation levels to labels
labels = {
    0: "not_curated",
    1: "library",
    2: "playlist",
    3: "both"
}

history_all["curation_label"] = history_all["curation_level"].map(labels)


### Aggregation and Preprocessing

In [7]:
# Aggregate by track and user
track_summary = (
    history_all
    .groupby(["user_id", "match_key"])
    .agg({
        "msPlayed": "sum",
        "trackName": "first",
        "artistName": "first",
        "curation_level": "max",
        "curation_label": "first",
        "hour": "mean"
    })
    .reset_index()
)

# Convert to minutes
track_summary["minutes_played"] = track_summary["msPlayed"] / 60000

# Adding playcount as metric
play_counts = (
    history_all
    .groupby(["user_id", "match_key"])
    .size()
    .reset_index(name="play_count")
)

track_summary = track_summary.merge(
    play_counts,
    on=["user_id", "match_key"],
    how="left"
)

# Load and prepare track metadata
track_meta = pd.read_csv("data/track_data.csv")

track_meta["artist_clean"] = track_meta["artists"].apply(clean_text)
track_meta["track_clean"] = track_meta["track_name"].apply(clean_text)

track_meta["match_key"] = track_meta["artist_clean"] + "_" + track_meta["track_clean"]

full_data = pd.merge(
    track_summary,
    track_meta,
    on="match_key",
    how="left"
)

print("Final dataset shape:", full_data.shape)

Final dataset shape: (48676, 34)
